# Filling Missing Rushing Yards: Mean vs Team Mean vs Model
**Juan Zavala** · AI for Analytics · `season_stats.csv`, 272 games, one row per game (the winning team)

Claude worked as a **pair programmer**: each part starts with a plain-English discussion, the plan is reviewed, and only then is code written and run.

| Label | Meaning |
|---|---|
| **[ME]** | Code I wrote myself |
| **[CLAUDE]** | Code Claude wrote after I approved its plan |
| **[REVIEW]** | What I changed or questioned in Claude's plan or code |
| **[CHECK]** | A validation step: does the code do what we think it does? |

The Part 1 discussion (before any code) is in [`part1_exploration.md`](part1_exploration.md).

**The answer key stays out of the analysis.** `actual.csv` is first read in Part 7. Everything before that uses only `season_stats.csv`. The method choices and the model-selection rule were fixed before modeling, and the expected errors are committed to git before Part 7 runs.

---
# Part 1: The nulls, and what the null rows have in common
The assignment's section 2 asks for the data to be read in as `stats`, so the exploration uses that name too.

In [1]:
# [ME] Load the data. The first CSV column is an unnamed row ID (12 to 283), so it becomes the index.
import pandas as pd
import numpy as np

pd.set_option('display.width', 200)
stats = pd.read_csv('data/season_stats.csv', index_col=0)
print(stats.shape)
stats.head()

(272, 7)


,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
12,18,Cowboys,Football Team,38,10,131.0,309
13,18,Rams,49ers,21,20,109.0,163
14,18,Chiefs,Chargers,13,12,123.0,154
15,18,Raiders,Broncos,27,14,129.0,244
16,18,Titans,Jaguars,28,20,175.0,168


In [2]:
# [ME] Where are the nulls?
print(stats.isna().sum().to_string())
print()
print('Null rush_yds:', stats['rush_yds'].isna().sum(), 'of', len(stats),
      f"({stats['rush_yds'].isna().mean():.2%})")
print('rush_yds dtype:', stats['rush_yds'].dtype,
      '| all known values whole numbers:', (stats['rush_yds'].dropna() % 1 == 0).all())

week          0
name          0
opp_name      0
tm_score      0
opp_score     0
rush_yds     16
pass_yds      0

Null rush_yds: 16 of 272 (5.88%)
rush_yds dtype: float64 | all known values whole numbers: True


In [3]:
# [CLAUDE] What do the null rows have in common? Start with the week.
null_rows = stats['rush_yds'].isna()
print('Weeks of the null rows:', stats.loc[null_rows, 'week'].value_counts().to_dict())
print('Games in week 12:', (stats['week'] == 12).sum(),
      '| all of them null:', stats.loc[stats['week'] == 12, 'rush_yds'].isna().all(),
      '| any null outside week 12:', stats.loc[stats['week'] != 12, 'rush_yds'].isna().any())

# [CLAUDE] Something every row shares: is each row the winning team?
print('tm_score > opp_score in every row:', (stats['tm_score'] > stats['opp_score']).all(),
      '| ties:', (stats['tm_score'] == stats['opp_score']).sum())
print('(week, name) is unique:', not stats.duplicated(['week', 'name']).any())

stats.loc[null_rows, ['week', 'name', 'opp_name', 'tm_score', 'opp_score', 'rush_yds', 'pass_yds']]

Weeks of the null rows: {12: 16}
Games in week 12: 16 | all of them null: True | any null outside week 12: False
tm_score > opp_score in every row: True | ties: 0
(week, name) is unique: True


,week,name,opp_name,tm_score,opp_score,rush_yds,pass_yds
104,12,Bears,Vikings,12,10,NaN,217
105,12,Ravens,Chargers,20,10,NaN,177
106,12,Chiefs,Raiders,31,17,NaN,298
107,12,Eagles,Bills,37,34,NaN,200
108,12,Titans,Panthers,17,10,NaN,185
109,12,Giants,Patriots,10,7,NaN,191
110,12,Jaguars,Texans,24,21,NaN,364
111,12,Broncos,Browns,29,12,NaN,134
112,12,Rams,Cardinals,37,14,NaN,229
113,12,Colts,Buccaneers,27,20,NaN,251


In [4]:
# [CHECK] Is week 12 unusual in any other way? Compare it with the other 17 weeks.
from scipy import stats as sps

explore = stats.assign(margin=stats['tm_score'] - stats['opp_score'])
rows = []
for col in ['tm_score', 'opp_score', 'pass_yds', 'margin']:
    wk12, rest = explore.loc[null_rows, col], explore.loc[~null_rows, col]
    rows.append({'column': col, 'week 12 mean': round(wk12.mean(), 2), 'other weeks mean': round(rest.mean(), 2),
                 'Welch t-test p': round(sps.ttest_ind(wk12, rest, equal_var=False).pvalue, 3)})
print(pd.DataFrame(rows).set_index('column').to_string())

# [CHECK] Does rushing change across the season? (If it did, other weeks would be a biased stand-in.)
known = explore[~null_rows]
weekly = [g['rush_yds'].to_numpy() for _, g in known.groupby('week')]
rho = sps.spearmanr(known['week'], known['rush_yds'])
print(f"\nrush_yds vs week: Spearman r = {rho[0]:+.3f} (p = {rho[1]:.2f}) | "
      f"Kruskal-Wallis across weeks p = {sps.kruskal(*weekly).pvalue:.2f}")

           week 12 mean  other weeks mean  Welch t-test p
column                                                   
tm_score          26.44             27.46           0.686
opp_score         14.88             16.21           0.455
pass_yds         233.94            249.42           0.350
margin            11.56             11.25           0.895

rush_yds vs week: Spearman r = +0.025 (p = 0.69) | Kruskal-Wallis across weeks p = 0.12


**What the null rows have in common:** all 16 are in **week 12**, and every week-12 game is null. One whole week of rushing totals is missing, and no other week has a single gap.

Two more facts apply to the whole file:

* **Every row is the winning team.** `tm_score` beats `opp_score` in all 272 rows. So the 16 missing games are wins, and a team's average from its other wins is a like-for-like stand-in.
* **`rush_yds` is a decimal column only because of the nulls.** Every known value is a whole number, which is why the fills must keep it an integer.

**Is week 12 otherwise unusual?** No. Its scores, passing yards and score margins all match the other weeks (every p-value is above 0.3). Rushing yards don't drift across the season either (Spearman r = +0.025). The gap depends on the week alone, and the week is unrelated to rushing, so filling week 12 from the other 17 weeks shouldn't push the estimates up or down. Part 8 checks this against the answer key.

In [5]:
# [CLAUDE] Which columns carry information about rush_yds? (known rows only)
corr = pd.DataFrame({col: sps.pearsonr(known[col], known['rush_yds']) for col in
                     ['tm_score', 'margin', 'pass_yds', 'opp_score', 'week']}, index=['pearson r', 'p-value']).T
print(corr.round(4).to_string())

team_groups = [g['rush_yds'].to_numpy() for _, g in known.groupby('name')]
opp_groups = [g['rush_yds'].to_numpy() for _, g in known.groupby('opp_name')]
team_share = known.groupby('name')['rush_yds'].transform('mean').var(ddof=0) / known['rush_yds'].var(ddof=0)
print(f"\nTeam (name): ANOVA p = {sps.f_oneway(*team_groups).pvalue:.4f} | "
      f"share of variance explained in-sample = {team_share:.3f}")
print(f"Opponent (opp_name): ANOVA p = {sps.f_oneway(*opp_groups).pvalue:.4f}")
print(f"tm_score vs pass_yds (two of the predictors): r = {sps.pearsonr(known['tm_score'], known['pass_yds'])[0]:+.3f}")

# [CHECK] Leakage scan: no column may be computed from rush_yds
print('\nColumns:', list(stats.columns), '-> none is derived from rushing yards (no total-yards column).')

games_per_team = known['name'].value_counts()
print('\nKnown games per team: min', games_per_team.min(), '| max', games_per_team.max())
print('Known games for the 16 teams missing week 12:',
      games_per_team[stats.loc[null_rows, 'name']].sort_values().to_dict())

           pearson r  p-value
tm_score      0.3119   0.0000
margin        0.2733   0.0000
pass_yds     -0.2093   0.0008
opp_score     0.0199   0.7519
week          0.0012   0.9853

Team (name): ANOVA p = 0.0149 | share of variance explained in-sample = 0.191
Opponent (opp_name): ANOVA p = 0.2549
tm_score vs pass_yds (two of the predictors): r = +0.384

Columns: ['week', 'name', 'opp_name', 'tm_score', 'opp_score', 'rush_yds', 'pass_yds'] -> none is derived from rushing yards (no total-yards column).

Known games per team: min 2 | max 12
Known games for the 16 teams missing week 12: {'Giants': 5, 'Titans': 5, 'Bears': 6, 'Falcons': 6, 'Broncos': 7, 'Jaguars': 8, 'Colts': 8, 'Packers': 8, 'Steelers': 9, 'Rams': 9, 'Chiefs': 10, 'Eagles': 10, 'Dolphins': 10, '49ers': 11, 'Cowboys': 11, 'Ravens': 12}


**What can predict the missing values?**

* **`tm_score` has the strongest link** (r = +0.31).
* **`pass_yds` has a negative one** (r = −0.21): teams that throw more tend to run less.
* **`opp_score` carries almost nothing on its own** (r = +0.02).
* **Team (`name`) matters** (ANOVA p = 0.015). The 19% "variance explained" is in-sample and optimistic, though, because some teams have very few games.
* **Opponent (`opp_name`) doesn't** (p = 0.25).

All of these relationships are modest, so no method will predict rushing yards closely. The question is which one gets closest.

**For the team mean:** the 16 teams missing week 12 have 5 to 12 known games each, so every one has a team average, and no fallback is needed. The smallest samples (Giants and Titans, 5 games) will be the noisiest.

---
# Part 2: Three copies to fill three different ways

In [6]:
# [ME] Section 2 of the assignment: stats2, stats3 and stats4 are independent copies of stats
stats2 = stats.copy()
stats3 = stats.copy()
stats4 = stats.copy()

# [CHECK] Independent copies: no copy shares memory with stats, so filling one can't change another
for name, df in [('stats2', stats2), ('stats3', stats3), ('stats4', stats4)]:
    print(name, '| same object as stats:', df is stats,
          '| shares memory with stats:', np.shares_memory(df['rush_yds'].to_numpy(), stats['rush_yds'].to_numpy()))

# The 16 rows every method must fill (and the only rows that will be scored in Part 8)
fill_rows = stats.index[stats['rush_yds'].isna()]
print('Rows to fill:', len(fill_rows), '| IDs', fill_rows.min(), 'to', fill_rows.max())

stats2 | same object as stats: False | shares memory with stats: False
stats3 | same object as stats: False | shares memory with stats: False
stats4 | same object as stats: False | shares memory with stats: False
Rows to fill: 16 | IDs 104 to 119


---
# Part 4: The simple solution, the overall mean (`stats2`)

### Prompt 1
> "Give me pandas code to replace the null values in the `rush_yds` column with the average of the non-null values in that column."

### Prompt 2, altered so the column stays an integer
> "Same thing, but the `rush_yds` column must stay an int, because yards are whole numbers."

**Claude's approach:** take the average of the 256 known values, round it to a whole number of yards, fill the 16 nulls with it, then convert the column to `int`.

In [7]:
# [CLAUDE] Prompt 1's answer, previewed on a copy so stats2 isn't changed yet
preview = stats2['rush_yds'].fillna(stats2['rush_yds'].mean())
print('dtype after prompt 1:', preview.dtype, '| value used:', preview.loc[fill_rows].unique())

dtype after prompt 1: float64 | value used: [126.89453125]


**[REVIEW]** Prompt 1 fills every gap with 126.89453125 yards, and the column stays a decimal (`float64`). Real rushing totals are whole numbers, which is why prompt 2 asks for an int.

In [8]:
# [CLAUDE] Prompt 2's answer: round the mean to a whole number of yards, fill, then convert to int
mean_yds = round(stats2['rush_yds'].mean())          # 126.89 -> 127
stats2['rush_yds'] = stats2['rush_yds'].fillna(mean_yds).astype(int)

# [CHECK] No nulls left, the column is an integer, and the 256 known values are untouched
assert stats2['rush_yds'].isna().sum() == 0
assert pd.api.types.is_integer_dtype(stats2['rush_yds'])
assert (stats2['rush_yds'].drop(fill_rows) == stats['rush_yds'].drop(fill_rows)).all()
print('Fill value:', mean_yds, '| dtype:', stats2['rush_yds'].dtype)

# [CHECK] Is 127 a reasonable rushing total? Compare it with the known games.
known_yds = stats['rush_yds'].dropna()
print('Known games: median', known_yds.median(), '| middle half', known_yds.quantile(0.25), 'to', known_yds.quantile(0.75),
      f'| 127 is at the {(known_yds < mean_yds).mean():.0%} percentile')
print('Spread (standard deviation): known games', round(known_yds.std(), 1), '| all 272 after filling', round(stats2['rush_yds'].std(), 1))
stats2.loc[fill_rows, ['week', 'name', 'opp_name', 'tm_score', 'rush_yds']].head(4)

Fill value: 127 | dtype: int64
Known games: median 124.0 | middle half 94.5 to 155.25 | 127 is at the 53% percentile
Spread (standard deviation): known games 49.0 | all 272 after filling 47.5


,week,name,opp_name,tm_score,rush_yds
104,12,Bears,Vikings,12,127
105,12,Ravens,Chargers,20,127
106,12,Chiefs,Raiders,31,127
107,12,Eagles,Bills,37,127


**Is 127 reasonable?** As a single number, yes. It sits near the middle of the known games (median 124, middle half 94.5 to 155.25). But all 16 week-12 games get the same 127.

**Pros**
* The simplest possible fix: two lines of code.
* Part 1 showed the gap is unrelated to rushing, so filling with the mean doesn't pull the season average up or down.
* Nothing to tune, so nothing to overfit.

**Cons**
* Ignores everything about the game: who played, how many points they scored, how much they passed.
* Sixteen identical values shrink the spread. The standard deviation drops from 49.0 (known games) to 47.5 (all 272 rows), so any later look at variability would understate it.
* It can't produce a big rushing day or a bad one. Every guess is "average".

---
# Part 5: Slightly more complex, the team mean (`stats3`)

### Prompt
> "Give me a Python script that replaces the nulls in `rush_yds` with that team's average (the team is in the `name` column). The column must stay an int."

**Claude's approach:**
1. Use `groupby('name')['rush_yds'].transform('mean')` to put each team's average on every one of its rows. This is one built-in pandas step, with no loop.
2. If a team had no known games, fall back to the overall average. Part 1 showed none of the 16 teams needs this, but the code shouldn't assume it.
3. Round to whole yards, fill, and convert to `int`.

In [9]:
# [CLAUDE] Team average, with an overall-average fallback, rounded to whole yards
team_avg = stats3.groupby('name')['rush_yds'].transform('mean')   # mean of each team's known games
fallback_needed = team_avg.isna() & stats3['rush_yds'].isna()
fill_values = team_avg.fillna(stats3['rush_yds'].mean()).round()

stats3['rush_yds'] = stats3['rush_yds'].fillna(fill_values).astype(int)

# [CHECK] No nulls, integer column, known values untouched, and how many rows needed the fallback
assert stats3['rush_yds'].isna().sum() == 0
assert pd.api.types.is_integer_dtype(stats3['rush_yds'])
assert (stats3['rush_yds'].drop(fill_rows) == stats['rush_yds'].drop(fill_rows)).all()
print('Rows that needed the overall-average fallback:', int(fallback_needed.sum()))
print('Team averages that end in exactly .5 (where the rounding rule would matter):',
      int(((team_avg.loc[fill_rows] * 2) % 2 == 1).sum()))

games = stats['rush_yds'].notna().groupby(stats['name']).sum()
filled3 = stats3.loc[fill_rows, ['week', 'name', 'rush_yds']].assign(
    team_average=team_avg.loc[fill_rows].round(2), known_games=stats.loc[fill_rows, 'name'].map(games))
filled3

Rows that needed the overall-average fallback: 0
Team averages that end in exactly .5 (where the rounding rule would matter): 0


,week,name,rush_yds,team_average,known_games
104,12,Bears,178,178.00,6
105,12,Ravens,162,161.75,12
106,12,Chiefs,108,108.00,10
107,12,Eagles,137,137.10,10
108,12,Titans,144,144.20,5
109,12,Giants,128,128.40,5
110,12,Jaguars,124,124.12,8
111,12,Broncos,110,110.00,7
112,12,Rams,131,131.11,9
113,12,Colts,121,120.62,8


In [10]:
# [CHECK] Recompute three teams' averages by hand, with plain Python, from their known games
for team in ['Giants', 'Bears', 'Ravens']:
    values = stats.loc[(stats['name'] == team) & stats['rush_yds'].notna(), 'rush_yds'].astype(int).tolist()
    by_hand = sum(values) / len(values)
    print(f'{team}: {values} -> {sum(values)} / {len(values)} = {by_hand:.2f} -> filled with',
          stats3.loc[(stats3['name'] == team) & (stats3['week'] == 12), 'rush_yds'].item())

Giants: [109, 209, 91, 106, 127] -> 642 / 5 = 128.40 -> filled with 128
Bears: [192, 250, 142, 133, 173, 178] -> 1068 / 6 = 178.00 -> filled with 178
Ravens: [160, 102, 251, 139, 157, 298, 130, 146, 139, 131, 178, 110] -> 1941 / 12 = 161.75 -> filled with 162


**Are the team averages reasonable?** Yes. They range from 108 (Chiefs) to 178 (Bears), all well inside the normal range. The hand check above matches exactly for the Giants, Bears and Ravens, and no team needed the fallback.

**Pros**
* Uses team identity, which carries real signal (Part 1: ANOVA p = 0.015).
* Still one line of pandas.
* The averages come from wins, the same kind of games as the missing ones.

**Cons**
* **Small samples.** The Giants' and Titans' averages each rest on 5 games, so one unusual game moves them a lot. The Bears' 178 includes a 250-yard game, and without it their average would be 164.
* Still ignores the game itself: the score and the passing yards.
* A team with no known games would need a fallback. That doesn't happen here, but the code handles it.

Part 6 tests whether team identity is worth this noise.

---
# Part 6: Most complex, a model (`stats4`)

### Prompt
> "Give me code that replaces the nulls in `rush_yds` with values predicted from the `name`, `tm_score`, `opp_score` and `pass_yds` columns, keeping the column an int. Explain the plan first and list a few candidate models."

### Claude's plan (fixed before any model was run)
1. **Features:** exactly the four the assignment names. `name` is text, so it becomes 32 yes/no team columns (one-hot encoding). The three numbers are standardized.
2. **Candidates:** linear regression, ridge regression, random forest and gradient boosting. Each is a scikit-learn `Pipeline`, so the preprocessing is learned only from training rows. Settings are fixed in advance; ridge picks its own penalty strength *inside* each training fold.
3. **How to compare them:** leave-one-week-out cross-validation on the 256 known rows. Each of the 17 known weeks is hidden in turn and predicted from the other 16. This is exactly the real problem, where a whole week (week 12) is missing. The overall mean and team mean go through the same test, so all methods are compared on equal terms.
4. **Selection rule:** take the lowest average error per hidden week (MAE). If another candidate is within one standard error of it, take the simpler model.
5. **Fill:** fit the chosen model on all 256 known rows, predict the 16 week-12 games, round to whole yards, and convert to `int`.

**[REVIEW]**
* I asked why the comparison doesn't randomly shuffle the rows into folds. Hiding whole weeks copies how the data is actually missing. A random split would let a model learn from the same week it is predicting, which never happens with week 12. A random 5-fold check is still run below as a second opinion.
* I also confirmed the answer key isn't used anywhere here.

In [11]:
# [CLAUDE] Candidate models: scikit-learn pipelines on the four assignment columns
import warnings
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

FEATURES = ['name', 'tm_score', 'opp_score', 'pass_yds']
TEXT_COLUMNS = {'name', 'opp_name'}

def preprocess(columns):
    text = [c for c in columns if c in TEXT_COLUMNS]
    numbers = [c for c in columns if c not in TEXT_COLUMNS]
    steps = []
    if text:
        steps.append(('teams', OneHotEncoder(handle_unknown='ignore', sparse_output=False), text))
    if numbers:
        steps.append(('numbers', StandardScaler(), numbers))
    return ColumnTransformer(steps)

CANDIDATES = {
    'Linear regression': lambda cols: make_pipeline(preprocess(cols), LinearRegression()),
    'Ridge regression': lambda cols: make_pipeline(preprocess(cols), RidgeCV(alphas=np.logspace(-2, 3, 26))),
    'Random forest': lambda cols: make_pipeline(preprocess(cols), RandomForestRegressor(
        n_estimators=500, min_samples_leaf=5, random_state=42)),
    'Gradient boosting': lambda cols: make_pipeline(preprocess(cols), HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.05, min_samples_leaf=10, random_state=42)),
}
known_rows = stats[stats['rush_yds'].notna()]
print('Training rows:', len(known_rows), '| known weeks:', known_rows['week'].nunique())

Training rows: 256 | known weeks: 17


In [12]:
# [CLAUDE] Leave-one-week-out: hide each known week, predict it from the other 16 weeks
def leave_one_week_out(predict, data=known_rows):
    predictions = pd.Series(np.nan, index=data.index)
    for week in sorted(data['week'].unique()):
        train, test = data[data['week'] != week], data[data['week'] == week]
        predictions.loc[test.index] = predict(train, test)
    errors = predictions - data['rush_yds']
    per_week = pd.DataFrame({'week': data['week'], 'abs_err': errors.abs(), 'sq_err': errors ** 2}).groupby('week').mean()
    return predictions, per_week

def overall_mean(train, test):
    return np.full(len(test), train['rush_yds'].mean())

def team_mean(train, test):
    by_team = train.groupby('name')['rush_yds'].mean()
    return test['name'].map(by_team).fillna(train['rush_yds'].mean()).to_numpy()

def model_predictor(make_model, columns=FEATURES):
    return lambda train, test: make_model(columns).fit(train[columns], train['rush_yds']).predict(test[columns])

methods = {'Overall mean (stats2 method)': overall_mean, 'Team mean (stats3 method)': team_mean}
methods.update({name: model_predictor(make) for name, make in CANDIDATES.items()})

cv_predictions, cv_rows = {}, []
for name, predict in methods.items():
    preds, per_week = leave_one_week_out(predict)
    cv_predictions[name] = preds
    n = len(per_week)
    cv_rows.append({'method': name,
                    'MAE per hidden week': per_week['abs_err'].mean(), 'MAE std. error': per_week['abs_err'].std(ddof=1) / np.sqrt(n),
                    'MSE per hidden week': per_week['sq_err'].mean(), 'MSE std. error': per_week['sq_err'].std(ddof=1) / np.sqrt(n)})
cv_table = pd.DataFrame(cv_rows).set_index('method').sort_values('MAE per hidden week')
cv_table.round(2)

,MAE per hidden week,MAE std. error,MSE per hidden week,MSE std. error
method,,,,
Ridge regression,34.09,2.01,1932.22,219.41
Linear regression,36.43,1.92,2140.15,221.13
Random forest,37.02,2.05,2294.76,282.45
Overall mean (stats2 method),37.62,2.29,2391.29,300.56
Team mean (stats3 method),39.79,2.06,2502.01,263.64
Gradient boosting,40.56,2.35,2822.39,311.55


In [13]:
# [CLAUDE] Apply the selection rule written down before any model was run
complexity = ['Linear regression', 'Ridge regression', 'Random forest', 'Gradient boosting']   # simplest first
models_only = cv_table.loc[complexity]
best = models_only['MAE per hidden week'].idxmin()
limit = models_only.loc[best, 'MAE per hidden week'] + models_only.loc[best, 'MAE std. error']
within_one_se = [m for m in complexity if models_only.loc[m, 'MAE per hidden week'] <= limit]
chosen = within_one_se[0]                     # the simplest model within one standard error of the best
print(f'Lowest error: {best} ({models_only.loc[best, "MAE per hidden week"]:.2f}). '
      f'One-standard-error limit: {limit:.2f}. Within it: {within_one_se}.')
print('Chosen model:', chosen)

Lowest error: Ridge regression (34.09). One-standard-error limit: 36.10. Within it: ['Ridge regression'].
Chosen model: Ridge regression


**Result:** ridge regression has the lowest error per hidden week, **34.09 yards**. No other model is within one standard error of it (the limit is 36.10), so **ridge is the chosen model**. Three findings stand out:

* **The team mean (39.79) is *worse* than the plain overall mean (37.62).** Team averages built from a handful of games carry so much noise that they hurt more than they help.
* **More complex isn't automatically better.** Gradient boosting is the least accurate method of all (40.56), and the random forest (37.02) barely beats the overall mean. With only 256 rows, flexible models chase noise.
* **Ridge beats plain linear regression (36.43) using the same four columns.** The only difference is that ridge shrinks the 32 team adjustments toward zero, which is exactly what thin team samples need.

In [14]:
# [CLAUDE] Fit the chosen model on all 256 known rows and predict the 16 week-12 games
final_model = CANDIDATES[chosen](FEATURES).fit(known_rows[FEATURES], known_rows['rush_yds'])
predicted = pd.Series(final_model.predict(stats4.loc[fill_rows, FEATURES]), index=fill_rows)
stats4['rush_yds'] = stats4['rush_yds'].fillna(predicted.round()).astype(int)

# [CHECK] No nulls, integer column, known values untouched, predictions in a plausible range
assert stats4['rush_yds'].isna().sum() == 0
assert pd.api.types.is_integer_dtype(stats4['rush_yds'])
assert (stats4['rush_yds'].drop(fill_rows) == stats['rush_yds'].drop(fill_rows)).all()
print('Ridge penalty chosen on the training rows (alpha):', round(final_model[-1].alpha_, 2))
print('Predictions range from', int(stats4.loc[fill_rows, 'rush_yds'].min()), 'to', int(stats4.loc[fill_rows, 'rush_yds'].max()),
      '| known games range from', int(known_yds.min()), 'to', int(known_yds.max()))
stats4.loc[fill_rows, ['week', 'name', 'opp_name', 'tm_score', 'opp_score', 'pass_yds', 'rush_yds']]

Ridge penalty chosen on the training rows (alpha): 25.12
Predictions range from 91 to 159 | known games range from 29 to 350


,week,name,opp_name,tm_score,opp_score,pass_yds,rush_yds
104,12,Bears,Vikings,12,10,217,108
105,12,Ravens,Chargers,20,10,177,134
106,12,Chiefs,Raiders,31,17,298,121
107,12,Eagles,Bills,37,34,200,159
108,12,Titans,Panthers,17,10,185,122
109,12,Giants,Patriots,10,7,191,104
110,12,Jaguars,Texans,24,21,364,91
111,12,Broncos,Browns,29,12,134,153
112,12,Rams,Cardinals,37,14,229,155
113,12,Colts,Buccaneers,27,20,251,124


In [15]:
# [CHECK] What did the model learn? Convert the standardized coefficients back to plain units.
names = final_model[0].get_feature_names_out()
coefs = pd.Series(final_model[-1].coef_, index=names)
scales = final_model[0].named_transformers_['numbers'].scale_
for col, scale in zip(['tm_score', 'opp_score', 'pass_yds'], scales):
    print(f"{col:10s}: {coefs['numbers__' + col] / scale:+.2f} rushing yards per 1 more unit")
team_effects = coefs[[n for n in names if n.startswith('teams__')]].rename(lambda n: n.replace('teams__name_', ''))
print('\nTeam adjustments (yards, after ridge shrinkage): smallest', team_effects.idxmin(), round(team_effects.min(), 1),
      '| largest', team_effects.idxmax(), round(team_effects.max(), 1))

tm_score  : +2.18 rushing yards per 1 more unit
opp_score : -0.10 rushing yards per 1 more unit
pass_yds  : -0.22 rushing yards per 1 more unit

Team adjustments (yards, after ridge shrinkage): smallest Chargers -7.7 | largest Bears 7.3


**Are the predictions reasonable?** Yes. The 16 predictions run from 91 to 159 yards, inside the range of known games (29 to 350). Unlike `stats2`'s sixteen identical 127s, they vary with the game. They are more cautious than real games, though: a model this uncertain pulls its guesses toward the middle.

**What the model learned, in plain units:**

* About **+2.2 rushing yards per point scored**.
* About **−0.22 per passing yard**, or roughly 22 fewer rushing yards per 100 passing yards.
* Almost nothing for the opponent's score (−0.10 per point).
* After shrinkage, the team adjustments stay small, from −7.7 (Chargers) to +7.3 (Bears).

**Pros**
* Uses the game's context.
* The most accurate method in the hidden-week test.
* Its coefficients can be read and sanity-checked.

**Cons**
* More code and more decisions: encoding, scaling, the penalty strength.
* Needs all four input columns for every row it predicts.
* Still overfits without the ridge penalty: plain linear regression was 2.34 yards worse (36.43 vs 34.09).
* Pulling toward the middle means it will underestimate the biggest rushing days.

### Extra (beyond the assignment): does every column earn its place?
The reflection asks why some columns are included and others left out, and whether Claude would choose well. So before running this test, Claude wrote down what it expected:

**Claude's prediction, written before the feature test below was run** (it was saved to a file and is preserved here unchanged):

1. `tm_score` will help the most, because it has the strongest correlation (r = +0.31).
2. `pass_yds` will also help, because of its negative relationship (run vs pass).
3. `opp_score` adds little on its own but may help alongside `tm_score`, since the two together give the score margin.
4. The 32 team columns (`name`) will add little, or even hurt a plain linear regression, because many teams have only 2 to 13 games. Regularization (ridge) should soften that.
5. Adding columns the assignment left out (`opp_name`, `week`, random noise) will make the cross-validated error **worse**, not better.

In [16]:
# [CHECK] Feature test: every combination of the four columns, plus extra columns the assignment left out.
# Everything uses leave-one-week-out on known rows only.
import itertools
rng = np.random.default_rng(42)
test_rows = known_rows.copy()
for i in range(10):
    test_rows['noise_' + str(i)] = rng.normal(size=len(test_rows))      # columns of pure random noise
NOISE = ['noise_' + str(i) for i in range(10)]

def week_mae(model_name, columns):
    _, per_week = leave_one_week_out(model_predictor(CANDIDATES[model_name], columns), data=test_rows)
    return per_week['abs_err'].mean()

combo_rows = []
for k in range(1, 5):
    for combo in itertools.combinations(FEATURES, k):
        combo_rows.append({'columns': ' + '.join(combo), 'count': k,
                           'Ridge MAE': week_mae('Ridge regression', list(combo)),
                           'Linear MAE': week_mae('Linear regression', list(combo))})
extra_sets = {'4 columns + opp_name': FEATURES + ['opp_name'], '4 columns + week': FEATURES + ['week'],
              '4 columns + 10 noise columns': FEATURES + NOISE, '4 columns + opp_name + week + 10 noise': FEATURES + ['opp_name', 'week'] + NOISE}
for label, columns in extra_sets.items():
    combo_rows.append({'columns': label, 'count': len(columns),
                       'Ridge MAE': week_mae('Ridge regression', columns), 'Linear MAE': week_mae('Linear regression', columns)})
feature_test = pd.DataFrame(combo_rows).set_index('columns')
feature_test.round(2)

,count,Ridge MAE,Linear MAE
columns,,,
name,1,37.82,39.79
tm_score,1,36.10,36.09
opp_score,1,37.62,37.68
pass_yds,1,37.12,37.06
name + tm_score,2,36.25,38.53
name + opp_score,2,38.02,40.11
name + pass_yds,2,36.90,38.67
tm_score + opp_score,2,36.36,36.43
tm_score + pass_yds,2,34.23,34.25


**How Claude's prediction held up:**

| # | Prediction (written before the test) | What the test showed | Verdict |
|---|---|---|---|
| 1 | `tm_score` helps the most | Best single column (ridge 36.10) | Right |
| 2 | `pass_yds` also helps | Adding it to `tm_score` drops the error to 34.23 | Right |
| 3 | `opp_score` helps alongside `tm_score` | `tm_score + opp_score` (36.36) is no better than `tm_score` alone (36.10) | **Wrong** |
| 4 | Team columns add little; they hurt plain linear regression | Ridge: 34.23 → 34.06 with `name`. Linear: 34.25 → 36.23 | Right |
| 5 | Extra columns make it worse | Ridge: + `week` 34.34, + noise 34.67, + all extras 34.88 (vs 34.09). + `opp_name` 34.06, no real change. Linear: every extra set is worse, up to 40.19 | Mostly right |

So Claude reasoned well about the columns but got one wrong, and only the test showed which one.

**Too many columns:** with plain linear regression, extra columns steadily hurt, from 36.43 with the four assignment columns to 40.19 with 16 columns. Ridge holds up (34.09 → 34.88) because its penalty shrinks useless columns toward zero. And the four assignment columns aren't the best set: `name + tm_score + pass_yds` (34.06), and even `tm_score + pass_yds` alone (34.23), are essentially as good as all four.

In [17]:
# [CHECK] Second opinion on the ranking: random 5-fold cross-validation, repeated 5 times
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.dummy import DummyRegressor
second_opinion = {}
splitter = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
for name, model in [('Overall mean', make_pipeline(preprocess(FEATURES), DummyRegressor())),
                    ('Linear regression', CANDIDATES['Linear regression'](FEATURES)),
                    ('Ridge regression', CANDIDATES['Ridge regression'](FEATURES))]:
    scores = -cross_val_score(model, known_rows[FEATURES], known_rows['rush_yds'], cv=splitter, scoring='neg_mean_absolute_error')
    second_opinion[name] = round(float(scores.mean()), 2)
print('Random 5-fold MAE:', second_opinion)

Random 5-fold MAE: {'Overall mean': 37.77, 'Linear regression': 37.92, 'Ridge regression': 34.61}


A random 5-fold split, repeated 5 times, tells the same story: ridge 34.61, overall mean 37.77, plain linear regression 37.92. Ridge is the most accurate under both ways of testing.

---
## Pre-registered expectations (written before the answer key is opened)
The leave-one-week-out results are the best available estimate of how each method will do on week 12. They are saved to a file and committed to git **before** Part 7 reads `actual.csv`.

In [18]:
# [CLAUDE] Record each method's expected error on a hidden week, then save the file before Part 7
expected = pd.DataFrame({
    'dataframe': ['stats2', 'stats3', 'stats4'],
    'method': ['Overall mean', 'Team mean', chosen],
    'expected MAE': [cv_table.loc[m, 'MAE per hidden week'] for m in ['Overall mean (stats2 method)', 'Team mean (stats3 method)', chosen]],
    'expected MSE': [cv_table.loc[m, 'MSE per hidden week'] for m in ['Overall mean (stats2 method)', 'Team mean (stats3 method)', chosen]],
}).set_index('dataframe')
expected['expected rank (MAE)'] = expected['expected MAE'].rank().astype(int)
expected['expected rank (MSE)'] = expected['expected MSE'].rank().astype(int)
expected.round(2).to_csv('results/preregistered_expectations.csv')
expected.round(2)

,method,expected MAE,expected MSE,expected rank (MAE),expected rank (MSE)
dataframe,,,,,
stats2,Overall mean,37.62,2391.29,2,2
stats3,Team mean,39.79,2502.01,3,3
stats4,Ridge regression,34.09,1932.22,1,1


---
# Part 7: Reading in the actual data
This is the first time any code opens `actual.csv`. The expected errors above were committed to git before this cell existed (commit `87957e3`).

In [19]:
# [ME] Section 7: read the answer key
actual = pd.read_csv('data/actual.csv')
print(actual.shape, list(actual.columns))
actual.head()

(272, 3) ['week', 'name', 'actual_rush_yds']


,week,name,actual_rush_yds
0,18,Cowboys,131
1,18,Rams,109
2,18,Chiefs,123
3,18,Raiders,129
4,18,Titans,175


In [20]:
# [CHECK] Line the answer key up with season_stats by a real key (week + team), never by row order
key = ['week', 'name']
print('(week, name) is unique in actual:', not actual.duplicated(key).any())
lined_up = stats[key].merge(actual, on=key, how='left', validate='one_to_one')   # a left merge keeps stats' row order
truth = pd.Series(lined_up['actual_rush_yds'].to_numpy(), index=stats.index, name='actual_rush_yds')
assert truth.notna().all(), 'every game in stats must have an answer'

known_mask = stats['rush_yds'].notna()
matches = int((truth[known_mask] == stats.loc[known_mask, 'rush_yds']).sum())
print(f'Games matched: {int(truth.notna().sum())} of {len(stats)} | '
      f'known games where actual equals season_stats exactly: {matches} of {int(known_mask.sum())}')

(week, name) is unique in actual: True
Games matched: 272 of 272 | known games where actual equals season_stats exactly: 256 of 256


**The answer key lines up.** `(week, name)` is unique, and all 272 games matched. On the 256 games where `season_stats` already had a value, the answer key agrees exactly (256 of 256). That proves the rows are paired correctly, by key rather than by row order, before anything is scored.

---
# Part 8: Testing the results

### Prompt for mean absolute error
> "Give me code that computes the mean absolute error between `rush_yds` and `actual_rush_yds`, only on the rows where I replaced null values."

(The assignment says "ChatGPT" here; Claude was used throughout.)

**[REVIEW]** "Only on the rows where I replaced null values" is essential. The other 256 rows are copies of the true values, so including them would add 256 zero-errors and make every method look about 16 times better than it is.

In [21]:
# [CLAUDE] Mean absolute error on the 16 filled rows only
from sklearn.metrics import mean_absolute_error, mean_squared_error

filled = {'stats2 (overall mean)': stats2, 'stats3 (team mean)': stats3, 'stats4 (ridge model)': stats4}
true_yds = truth.loc[fill_rows]

mae = {}
for label, df in filled.items():
    guess = df.loc[fill_rows, 'rush_yds']
    mae[label] = mean_absolute_error(true_yds, guess)
    assert np.isclose(mae[label], np.abs(guess - true_yds).mean())    # [CHECK] the same number computed by hand
mae = pd.Series(mae, name='MAE (yards)').sort_values()
print(mae.round(2).to_string())
print('\nBest (lowest MAE):', mae.idxmin(), '| Worst (highest MAE):', mae.idxmax())

stats4 (ridge model)     38.38
stats3 (team mean)       46.38
stats2 (overall mean)    48.88

Best (lowest MAE): stats4 (ridge model) | Worst (highest MAE): stats2 (overall mean)


### Prompt for mean squared error
> "Now give me code for the mean squared error on the same rows."

In [22]:
# [CLAUDE] Mean squared error on the same 16 rows (RMSE added, because it is back in yards)
mse = {}
for label, df in filled.items():
    guess = df.loc[fill_rows, 'rush_yds']
    mse[label] = mean_squared_error(true_yds, guess)
    assert np.isclose(mse[label], ((guess - true_yds) ** 2).mean())   # [CHECK] the same number computed by hand
scores = pd.DataFrame({'MAE': mae, 'MSE': pd.Series(mse)})
scores['RMSE'] = np.sqrt(scores['MSE'])
scores = scores.sort_values('MSE')
print(scores.round(2).to_string())
print('\nBest (lowest MSE):', scores['MSE'].idxmin(), '| Worst (highest MSE):', scores['MSE'].idxmax())

                         MAE      MSE   RMSE
stats4 (ridge model)   38.38  1972.38  44.41
stats3 (team mean)     46.38  2697.00  51.93
stats2 (overall mean)  48.88  3052.38  55.25

Best (lowest MSE): stats4 (ridge model) | Worst (highest MSE): stats2 (overall mean)


### The answers

| | Best | Worst |
|---|---|---|
| **Mean absolute error** | **stats4, ridge model: 38.38 yards** | stats2, overall mean: 48.88 yards |
| **Mean squared error** | **stats4, ridge model: 1,972.38** | stats2, overall mean: 3,052.38 |

The team mean (`stats3`) lands in between on both: 46.38 and 2,697.00.

**The two metrics tell the same story, with one difference.** The model's lead is bigger on MSE (35% lower than the overall mean) than on MAE (21% lower). MSE squares each error, so it punishes big misses most, and the model avoided some of the biggest. For the Rams' 228-yard game, it missed by 73 yards where the overall mean missed by 101.

In [23]:
# [CHECK] Every filled game: the true value, each method's guess, and which guess was closest
games = stats.loc[fill_rows, ['name', 'opp_name', 'tm_score', 'pass_yds']].copy()
games['actual'] = true_yds
for label, short in zip(filled, ['mean', 'team mean', 'model']):
    games[short] = filled[label].loc[fill_rows, 'rush_yds']
for short in ['mean', 'team mean', 'model']:
    games[short + ' error'] = (games[short] - games['actual']).abs()
games['closest'] = games[['mean error', 'team mean error', 'model error']].idxmin(axis=1).str.replace(' error', '')
print('Closest guess, by game:', games['closest'].value_counts().to_dict())
games

Closest guess, by game: {'model': 7, 'team mean': 6, 'mean': 3}


,name,opp_name,tm_score,pass_yds,actual,mean,team mean,model,mean error,team mean error,model error,closest
104,Bears,Vikings,12,217,118,127,178,108,9,60,10,mean
105,Ravens,Chargers,20,177,197,127,162,134,70,35,63,team mean
106,Chiefs,Raiders,31,298,69,127,108,121,58,39,52,team mean
107,Eagles,Bills,37,200,185,127,137,159,58,48,26,model
108,Titans,Panthers,17,185,80,127,144,122,47,64,42,model
109,Giants,Patriots,10,191,58,127,128,104,69,70,46,model
110,Jaguars,Texans,24,364,81,127,124,91,46,43,10,model
111,Broncos,Browns,29,134,169,127,110,153,42,59,16,model
112,Rams,Cardinals,37,229,228,127,131,155,101,97,73,model
113,Colts,Buccaneers,27,251,155,127,121,124,28,34,31,mean


In [24]:
# [CHECK] How sure can we be with only 16 games? Bootstrap: resample the 16 games 10,000 times.
rng = np.random.default_rng(42)
resample = rng.integers(0, len(fill_rows), size=(10_000, len(fill_rows)))
errors = {label: (df.loc[fill_rows, 'rush_yds'] - true_yds).to_numpy() for label, df in filled.items()}

ci_rows = []
for label, e in errors.items():
    mae_draws, mse_draws = np.abs(e[resample]).mean(axis=1), (e[resample] ** 2).mean(axis=1)
    ci_rows.append({'method': label, 'MAE': np.abs(e).mean(), 'MAE 95% range': np.percentile(mae_draws, [2.5, 97.5]).round(1).tolist(),
                    'MSE': (e ** 2).mean(), 'MSE 95% range': np.percentile(mse_draws, [2.5, 97.5]).round(0).tolist()})
print(pd.DataFrame(ci_rows).set_index('method').round(2).to_string())

print('\nHead to head (same resampled games for both methods):')
labels = list(filled)
for a, b in [(labels[2], labels[0]), (labels[2], labels[1]), (labels[1], labels[0])]:
    diff = np.abs(errors[a][resample]).mean(axis=1) - np.abs(errors[b][resample]).mean(axis=1)
    print(f'{a} vs {b}: MAE difference {np.abs(errors[a]).mean() - np.abs(errors[b]).mean():+.2f} yards, '
          f'95% range {np.percentile(diff, [2.5, 97.5]).round(1).tolist()}, first is better in {np.mean(diff < 0):.1%} of resamples')

                         MAE MAE 95% range      MSE     MSE 95% range
method                                                               
stats2 (overall mean)  48.88  [37.0, 61.9]  3052.38  [1715.0, 4678.0]
stats3 (team mean)     46.38  [35.5, 58.3]  2697.00  [1537.0, 4037.0]
stats4 (ridge model)   38.38  [27.8, 49.6]  1972.38  [1055.0, 3056.0]

Head to head (same resampled games for both methods):
stats4 (ridge model) vs stats2 (overall mean): MAE difference -10.50 yards, 95% range [-18.8, -1.8], first is better in 99.0% of resamples
stats4 (ridge model) vs stats3 (team mean): MAE difference -8.00 yards, 95% range [-19.6, 3.9], first is better in 90.5% of resamples
stats3 (team mean) vs stats2 (overall mean): MAE difference -2.50 yards, 95% range [-11.5, 7.5], first is better in 70.2% of resamples


In [25]:
# [CHECK] Did the pre-registered estimates (committed before Part 7) call it?
pre = pd.read_csv('results/preregistered_expectations.csv', index_col='dataframe')
real = scores.rename(index=lambda s: s.split(' ')[0])
compare = pre[['method', 'expected MAE', 'expected rank (MAE)', 'expected MSE', 'expected rank (MSE)']].assign(
    **{'actual MAE': real['MAE'].round(2), 'actual rank (MAE)': real['MAE'].rank().astype(int),
       'actual MSE': real['MSE'].round(2), 'actual rank (MSE)': real['MSE'].rank().astype(int)})
compare

,method,expected MAE,expected rank (MAE),expected MSE,expected rank (MSE),actual MAE,actual rank (MAE),actual MSE,actual rank (MSE)
dataframe,,,,,,,,,
stats2,Overall mean,37.62,2,2391.29,2,48.88,3,3052.38,3
stats3,Team mean,39.79,3,2502.01,3,46.38,2,2697.00,2
stats4,Ridge regression,34.09,1,1932.22,1,38.38,1,1972.38,1


In [26]:
# [CHECK] Why were all three methods worse than their estimates? Look at week 12 itself.
weekly_mean = truth.groupby(stats['week']).mean()
order = weekly_mean.sort_values(ascending=False)
print(f'Week 12 average rushing: {weekly_mean[12]:.1f} yards | rank {list(order.index).index(12) + 1} of 18 weeks | '
      f'other weeks range {weekly_mean.drop(12).min():.1f} to {weekly_mean.drop(12).max():.1f}')
w12, rest = truth[stats['week'] == 12], truth[stats['week'] != 12]
print(f'Week 12 vs the other weeks: Welch t-test p = {sps.ttest_ind(w12, rest, equal_var=False).pvalue:.2f}')
for label, e in errors.items():
    print(f'{label}: average signed error {e.mean():+.2f} yards (negative = guessed too low)')

Week 12 average rushing: 141.6 yards | rank 5 of 18 weeks | other weeks range 102.8 to 149.0
Week 12 vs the other weeks: Welch t-test p = 0.31
stats2 (overall mean): average signed error -14.62 yards (negative = guessed too low)
stats3 (team mean): average signed error -6.38 yards (negative = guessed too low)
stats4 (ridge model): average signed error -11.75 yards (negative = guessed too low)


In [27]:
# [CHECK] One week is a small test. Score all 18 weeks: the 17 known weeks from Part 6's
# leave-one-week-out predictions (no answer key needed), plus week 12 against the answer key.
cv_errors = pd.DataFrame({
    'stats2 (overall mean)': (cv_predictions['Overall mean (stats2 method)'] - known_rows['rush_yds']).abs(),
    'stats3 (team mean)': (cv_predictions['Team mean (stats3 method)'] - known_rows['rush_yds']).abs(),
    'stats4 (ridge model)': (cv_predictions[chosen] - known_rows['rush_yds']).abs(),
}).groupby(known_rows['week']).mean()
week12 = pd.DataFrame([mae], index=[12])
by_week = pd.concat([cv_errors, week12]).sort_index()
by_week.index.name = 'week'
by_week['lowest error'] = by_week[list(filled)].idxmin(axis=1)
print('Weeks won (lowest MAE), out of 18:', by_week['lowest error'].value_counts().to_dict())
print('Team mean beat the overall mean in', int((by_week[labels[1]] < by_week[labels[0]]).sum()), 'of 18 weeks')
print('The model beat the overall mean in', int((by_week[labels[2]] < by_week[labels[0]]).sum()), 'of 18 weeks, and the team mean in',
      int((by_week[labels[2]] < by_week[labels[1]]).sum()))
by_week.round(1)

Weeks won (lowest MAE), out of 18: {'stats4 (ridge model)': 15, 'stats3 (team mean)': 2, 'stats2 (overall mean)': 1}
Team mean beat the overall mean in 7 of 18 weeks
The model beat the overall mean in 16 of 18 weeks, and the team mean in 16


,stats2 (overall mean),stats3 (team mean),stats4 (ridge model),lowest error
week,,,,
1,38.6,42.3,38.4,stats4 (ridge model)
2,35.1,31.6,33.4,stats3 (team mean)
3,53.1,53.3,42.7,stats4 (ridge model)
4,26.2,39.9,31.7,stats2 (overall mean)
5,50.4,45.1,42.4,stats4 (ridge model)
6,42.5,42.7,35.5,stats4 (ridge model)
7,30.4,25.9,22.9,stats4 (ridge model)
8,41.8,46.6,38.2,stats4 (ridge model)
9,45.5,43.1,46.3,stats3 (team mean)


In [28]:
# [ME] Save the results for the submission
games.to_csv('results/imputed_comparison.csv')
pd.DataFrame(ci_rows).set_index('method').to_csv('results/method_scores.csv')
by_week.round(2).to_csv('results/weekly_method_errors.csv')
print('saved: results/imputed_comparison.csv, results/method_scores.csv, results/weekly_method_errors.csv')

saved: results/imputed_comparison.csv, results/method_scores.csv, results/weekly_method_errors.csv


## Part 8 conclusions

**1. The model (stats4) is the best method, with real evidence behind it.**
* It had the lowest MAE and the lowest MSE.
* Against the overall mean, the advantage holds up under resampling: the model was better in 99.0% of 10,000 resamples, and the 95% range of the difference (−18.8 to −1.8 yards) stays below zero.
* Against the team mean, it is likely better (90.5% of resamples), but that range (−19.6 to +3.9) includes zero.

**2. Team mean vs overall mean is a toss-up.**
* On week 12, the team mean was 2.5 yards better, but resampling can't separate the two (better in 70.2% of resamples, range −11.5 to +7.5).
* Across all 18 weeks, the team mean beat the overall mean in only 7. Which of these two simple methods "wins" depends on the week.

**3. The pre-registered prediction called the winner.**
* `stats4` was expected to rank first on both metrics, and it did.
* The other two swapped places, which matches how small and noisy their gap is.

**4. Every method did worse than its estimate, because week 12 was a heavy rushing week.**
* Week 12 averaged 141.6 rushing yards, 5th of 18 weeks. That's still inside the normal range (other weeks run 102.8 to 149.0, p = 0.31), so it's luck, not a pattern.
* All three methods guessed low on average (−14.6, −6.4 and −11.8 yards).
* Each method's week-12 error still falls inside the range it showed across the 17 hidden weeks. For the model, that range is 22.4 to 49.6, and week 12 came in at 38.4.

**5. One week is a small test, but the season agrees.**
* Scoring every week the same way (17 hidden known weeks plus week 12 against the answer key), the model had the lowest error in **15 of 18 weeks**.
* Game by game, it was the closest guess in 7 of the 16 week-12 games. Even the best method isn't best every time.

**What no method could do:** predict the big days. The Rams and Falcons each rushed for 228 yards in week 12, and every method missed both games by 73 yards or more.

---
# Part 9: Reflection
The five reflection questions are answered in [`reflection.md`](reflection.md), the deliverable that gets emailed. Every answer there cites results from this notebook.